<a href="https://colab.research.google.com/github/carolinaaraujo-cap/C-lculo-Num-rico/blob/main/Anima%C3%A7%C3%A3o_de_gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba
from fractions import Fraction
from PIL import Image

# ---------------------------------------------------------------
# 1. Sistema e passos da eliminação (frações exatas)
# ---------------------------------------------------------------
A = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
b = [8, -11, -3]
M0 = [[Fraction(v) for v in row] + [Fraction(bb)] for row, bb in zip(A, b)]

def fmt(fr):
    return str(fr.numerator) if fr.denominator == 1 else f"{fr.numerator}/{fr.denominator}"

def op_text(t, p, m):
    if m > 0:
        return f"L{t+1} ← L{t+1} − ({fmt(m)})·L{p+1}"
    return f"L{t+1} ← L{t+1} + ({fmt(-m)})·L{p+1}"

def build_frames(M):
    M = [r[:] for r in M]
    frames = [dict(M=[r[:] for r in M], title="Sistema inicial (matriz aumentada)",
                   rows={}, cells={}, op="Três planos que se cruzam em um único ponto")]
    step = 1
    for p in range(2):
        for t in range(p + 1, 3):
            if M[t][p] == 0:
                continue
            m = M[t][p] / M[p][p]
            frames.append(dict(M=[r[:] for r in M],
                               title=f"Passo {step}: zerar a posição ({t+1},{p+1})",
                               rows={p: 'pivot_row', t: 'target_row'},
                               cells={(p, p): 'pivot', (t, p): 'zero'},
                               op=op_text(t, p, m)))
            M[t] = [a - m * c for a, c in zip(M[t], M[p])]
            frames.append(dict(M=[r[:] for r in M],
                               title=f"Passo {step}: resultado",
                               rows={t: 'target_row'}, cells={(t, p): 'done'},
                               op="A solução do sistema não mudou"))
            step += 1
    # substituição regressiva
    z = M[2][3] / M[2][2]
    y = (M[1][3] - M[1][2] * z) / M[1][1]
    x = (M[0][3] - M[0][1] * y - M[0][2] * z) / M[0][0]
    frames.append(dict(M=[r[:] for r in M],
                       title=f"Solução: x = {fmt(x)},  y = {fmt(y)},  z = {fmt(z)}",
                       rows={}, cells={(0, 3): 'done', (1, 3): 'done', (2, 3): 'done'},
                       op=f"Substituição regressiva:  z = {fmt(z)}  ⇒  y = {fmt(y)}  ⇒  x = {fmt(x)}"))
    return frames, (float(x), float(y), float(z))

frames, SOL = build_frames(M0)

# ---------------------------------------------------------------
# 2. Estilo
# ---------------------------------------------------------------
COL = {'pivot': '#FFD54F', 'zero': '#EF5350', 'pivot_row': '#BBDEFB',
       'target_row': '#FFE0B2', 'done': '#A5D6A7'}
LEGEND = [('pivot', 'Pivô'), ('zero', 'Elemento a zerar'), ('pivot_row', 'Linha do pivô'),
          ('target_row', 'Linha de destino'), ('done', 'Resultado')]
PLANE_COL = ['#00ACC1', '#D81B60', '#7CB342']
LIM = 6

def plot_plane(ax, row, color, alpha):
    a, b_, c, d = [float(v) for v in row]
    coefs = np.array([a, b_, c])
    if np.abs(coefs).max() < 1e-12:
        return
    k = int(np.argmax(np.abs(coefs)))          # resolve pela variável de maior coeficiente
    g = np.linspace(-LIM, LIM, 30)
    U, V = np.meshgrid(g, g)
    if k == 2:   X, Y = U, V; Z = (d - a * X - b_ * Y) / c
    elif k == 1: X, Z = U, V; Y = (d - a * X - c * Z) / b_
    else:        Y, Z = U, V; X = (d - b_ * Y - c * Z) / a
    mask = (np.abs(X) > LIM) | (np.abs(Y) > LIM) | (np.abs(Z) > LIM)
    X, Y, Z = [np.where(mask, np.nan, arr) for arr in (X, Y, Z)]
    ax.plot_surface(X, Y, Z, color=color, alpha=alpha, linewidth=0, shade=False)

# ---------------------------------------------------------------
# 3. Renderização de um frame (tamanho FIXO, sem bbox_inches)
# ---------------------------------------------------------------
def render(fr, azim):
    fig = plt.figure(figsize=(13, 6.2), dpi=90, facecolor='white')
    fig.text(0.03, 0.92, fr['title'], fontsize=20, weight='bold', va='center')

    # --- tabela ---
    ax_t = fig.add_axes([0.03, 0.40, 0.50, 0.40]); ax_t.axis('off')
    text = [[fmt(v) for v in row] for row in fr['M']]
    tbl = ax_t.table(cellText=text, colLabels=['x', 'y', 'z', 'b'],
                     rowLabels=['L1', 'L2', 'L3'], cellLoc='center',
                     loc='center', bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(20)
    for c in range(4):
        tbl[(0, c)].set_facecolor('#ECEFF1'); tbl[(0, c)].set_text_props(weight='bold')
    for r in range(3):
        tbl[(r + 1, -1)].set_facecolor(to_rgba(PLANE_COL[r], 0.45))  # cor = cor do plano
        tbl[(r + 1, -1)].set_text_props(weight='bold')
        for c in range(4):
            tbl[(r + 1, c)].set_facecolor('white')
            tbl[(r + 1, c)].set_edgecolor('#607D8B')
    for r, kind in fr['rows'].items():
        for c in range(4):
            tbl[(r + 1, c)].set_facecolor(COL[kind])
    for (r, c), kind in fr['cells'].items():
        tbl[(r + 1, c)].set_facecolor(COL[kind])
        tbl[(r + 1, c)].set_text_props(weight='bold')

    # --- operação ---
    fig.text(0.28, 0.27, fr['op'], ha='center', va='center', fontsize=17,
             color='#B71C1C',
             bbox=dict(boxstyle='round,pad=0.5', fc='#FFF8E1', ec='#9E9E9E'))

    # --- legenda ---
    fig.legend(handles=[mpatches.Patch(fc=COL[k], ec='gray', label=l) for k, l in LEGEND],
               loc='lower left', bbox_to_anchor=(0.02, 0.02), ncol=3, fontsize=12,
               frameon=True)

    # --- 3D ---
    ax = fig.add_axes([0.55, 0.03, 0.44, 0.90], projection='3d')
    active = set(fr['rows'].keys())
    for i in range(3):
        alpha = 0.55 if (not active or i in active) else 0.15
        plot_plane(ax, fr['M'][i][:], PLANE_COL[i], alpha)
    ax.scatter(*SOL, color='red', s=140, marker='*', depthshade=False, zorder=10)
    ax.text(SOL[0] + 0.3, SOL[1] + 0.3, SOL[2] + 0.8,
            f"({SOL[0]:g}, {SOL[1]:g}, {SOL[2]:g})", color='red', fontsize=13, weight='bold')
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_zlim(-LIM, LIM)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.view_init(elev=22, azim=azim)
    ax.legend(handles=[mpatches.Patch(fc=PLANE_COL[i], alpha=0.6, label=f'L{i+1}') for i in range(3)],
              loc='upper left', fontsize=11)

    fig.canvas.draw()
    arr = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    plt.close(fig)
    return arr

# ---------------------------------------------------------------
# 4. Gera o GIF: cada passo vira vários sub-frames com a câmera girando
# ---------------------------------------------------------------
SUB = 12            # sub-frames por passo
MS = 150            # ms por sub-frame (~1,8 s por passo)
imgs = []
for i, fr in enumerate(frames):
    for s in range(SUB):
        azim = -60 + (i * SUB + s) * 1.2
        imgs.append(Image.fromarray(render(fr, azim)))

durations = [MS] * len(imgs)
durations[-1] = 4000            # pausa longa na solução final

pal = [im.convert('P', palette=Image.ADAPTIVE, colors=256) for im in imgs]
pal[0].save("gaussian_elimination_melhorado.gif", save_all=True, append_images=pal[1:],
            duration=durations, loop=0, disposal=2)
print(f"GIF salvo com {len(imgs)} quadros.")

In [ ]:
from google.colab import files
files.download('gaussian_elimination_melhorado.gif')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>